<center><h1>Hands-On Reinforcement Learning Applied to Trade Execution Algorithms</h1></center>
<center>
Autor: <cite><a href="https://www.linkedin.com/in/aoteog/">Oteo García, Alberto</a></cite>
</center>
<center>
Autor: <cite><a href="https://www.linkedin.com/in/jesus-sanz/">Sanz del Real, Jesús</a></cite>
</center>

----------------


<center><h1>Resumen</h1></center>

<center>

<p style="width:80%">
    En este notebook se presenta el taller de aprendizaje por refuerzo aplicado a un caso real para finanzas. 
    El objetivo es plantear, desde cero, la resolución del problema de <b>algoritmos de ejecución</b> a través del uso
    de técnicas de aprendizaje por refuerzo. Para ello, durante el taller se ha de completar el entorno básico <b>BestExecutionEnv</b> donde los asistentes desarrollarán las partes claves del
    código para obtener un simulador de mercado que, aunque relativamente naive, será funcional. Posteriormente, los participantes tendrán que aplicar algoritmos clásicos y basados en inteligencia artificial (IA) para la
    resolución del problema. 
</p>
<p style="width:80%">
    En términos sencillos, el problema de <b>algoritmos de ejecución</b> trata sobre la ejecución de órdenes con un tamaño lo suficientemente grande como para que su ejecución suponga un <b><i>'problema'</i></b>, como se ha comentado durante la explicación del caso práctico. La resolución de la ejecución de estas órdenes (órdenes CARE) se puede realizar utilizando diferentes aproximaciones. En el caso de este taller, queremos optimizar el <b><i>timing</i></b> de nuestras órdenes para minimizar el impacto de mercado. Es decir, buscaremos trocear la orden CARE para comprar (vender) a los menores (mayores) precios de la sesión en la que se ejecuta la orden CARE. Como baseline para saber si nuestra ejecución es buena, o mala, utilizaremos el baseline habitual en el sector: el VWAP slippage.
</p> 
<p style="width:80%">
El VWAP slippage es la desviación, en puntos básicos, del VWAP del mercado en comparación con el VWAP de nuestro algoritmo. Siendo el VWAP el precio medio ponderado por el volumen:
</p>   
    
$$\frac{\sum_{i=1}^T p_i\cdot v_i}{\sum_{i=1}^Tv_i}$$

</center>

---

<center><h1>Datos</h1></center>

<center>
<p style="width:80%">
Los datos para este taller se han extraído de los activos Repsol y Santander (<i>rep_data.pkl</i> y <i>san_data.pkl</i>). Entre la información extraída contamos con la primera posición del libro de órdenes, tanto del bid como del ask, así como el volumen acumulado, todo ello realizando una agrupación por 5 segundos. Las sesiones extraidas del libro de órdenes están comprendidas entre las 13:00h y las 17:25h. El conjunto de datos está partido en 3 subconjuntos, para los siguientes días bursátiles:
</p>
    
</center> 
<center>
    <p style="width:35%"><b>Train</b>: Primeros 60 días hábiles de 2018</p>
    <p style="width:35%"><b>Validación</b>: Primeros 30 días hábiles de 2019</p>
    <p style="width:35%"><b>Test</b>: Intervalo de los días hábiles del 31 al 60 de 2019 (30 días)</p>
</center>

<center>
<p style="width:80%">
<b>Adicionalmente</b>, para aquellos que quieran aumentar el grado de detalle, se han añadido otros dos ficheros con información más completa del libro de órdenes y de las ejecuciones de mercado. El fichero <i>orderbook.pkl</i> contiene las posiciones (precios y volúmenes) de hasta la tercera posición del libro de órdenes, agrupados por 5 segundos; el fichero <i>executions.pkl</i> todas las ejecuciones del mercado. En ambos casos, se trata de información para la compañía Repsol, entre las 13:00h y 17:25h de la sesión. El conjunto de datos está partido en 3 subconjuntos, para los siguientes días bursátiles:
</p>
    
</center> 
<center>
    <p style="width:35%"><b>Train</b>: Primeros 40 días hábiles de 2019</p>
    <p style="width:35%"><b>Validación</b>: Intervalo del 41 al 60 de 2019 (20 días)</p>
    <p style="width:35%"><b>Test</b>: Intervalo del 61 al 80 de 2019 (20 días)</p>
</center>

In [ ]:
import pickle
from typing import Tuple

import numpy as np
import pandas as pd

# Solo si `pip install -e .` o el Dev Container no te funcionan: descomenta estas
# dos líneas para añadir la raíz del repo al path y poder importar `src`
# import sys
# sys.path.append("..")

from src.agents.dqn import DDQNAgent
from src.report.report import plot_results
from src.rewards import VwapRewardStrategy

In [2]:
# Datos disponibles: san_data.pkl y rep_data.pkl
with open("../data/rep_data.pkl", "rb") as f:
        dict_ = pickle.load(f)
dict_.keys()

dict_keys(['train', 'val', 'test'])

In [3]:
data = dict_["train"]
data

{datetime.date(2018, 1, 2):                       ask1    bid1   vask1    vbid1    last     cumvol  \
 time                                                                     
 2018-01-02 13:00:00  15.00  14.995  1368.0   1878.0  15.000  1429424.0   
 2018-01-02 13:00:05  15.01  15.005  1308.0      1.0  15.000  1430792.0   
 2018-01-02 13:00:10  15.01  15.000  2988.0   5438.0  15.005  1431776.0   
 2018-01-02 13:00:15  15.01  15.000  2158.0   7941.0  15.005  1431776.0   
 2018-01-02 13:00:20  15.01  15.000  2138.0  10009.0  15.005  1431978.0   
 ...                    ...     ...     ...      ...     ...        ...   
 2018-01-02 17:24:40  14.98  14.970  1027.0   1574.0  14.970  4629270.0   
 2018-01-02 17:24:45  14.98  14.970  1027.0   1987.0  14.970  4629270.0   
 2018-01-02 17:24:50  14.98  14.970  1716.0   1921.0  14.975  4630053.0   
 2018-01-02 17:24:55  14.98  14.970  1844.0   1921.0  14.980  4630152.0   
 2018-01-02 17:25:00  14.98  14.970  1617.0   1921.0  14.980  4630152.0  

---
# Entorno

In [5]:
from src.environment.best_execution import BestExecutionEnv
from src.environment.config import EnvironmentConfig


---
<center><h2>TWAP</h2></center>

<center>
<p style="width:80%">
El <b><i>time weighted average price</i></b> (TWAP) es un algoritmo tradicional de negociación basado en el precio medio ponderado por el tiempo, utilizado normalmente para la ejecución de órdenes grandes para reducir el impacto de mercado. Básicamente, se trocea la orden de manera que ejecute las órdenes en un tiempo equidistante durante el tiempo de ejecución seleccionado por el cliente. Puede ser fácil adivinar el patrón de negociación de la estrategia en ejecución si sus órdenes no se modifican de una manera especial, por lo que los parámetros se pueden ajustar o alterar para hacer que la estrategia sea más difícil de rastrear. Las soluciones más comunes son la aleatorización del tamaño de las órdenes y/o el tiempo de envío entre cada una de ellas. Es posible limitar la cantidad para que no exceda un porcentaje definido del volumen de participación, o también evitar agresiones de más de una posición del libro. Esto nos ayuda a minimizar el impacto de las estrategias en el mercado.
</p>  
<p style="width:80%">
Para el objetivo del taller, se pide a los participantes implementar un TWAP sin la necesidad de realizar ninguna modificación. El agente TWAP ha de trocear y ejecutar las órdenes hijas de la orden CARE equidistantes en el tiempo, sin necesidad de incorporar mecanismos para evitar su detección.
</p>

In [ ]:
class TWAP(DDQNAgent):
    def act(self, s): # [T, V]
        # TODO: Configura un TWAP determinista utilizando s
        a = "-----" 
        return a

------------------
<center><h2>Parametrización del Problema</h2></center>

<center>
<p style="width:80%">
Una vez definido el entorno que vamos a utilizar como simulador de mercado, necesitamos definir el agente que va a encargarse de aprender a ejecutar las órdenes CARE. El agente es el <b><i>cerebro</i></b> del sistema y puede tener, o no, inteligencia artificial implementada. Por ejemplo, la clase TWAP anteriormente implementada depende únicamente de una regla determinista. Por otro lado, los algoritmos vistos durante el bloque de Aprendizaje por Refuerzo dependerán de la construcción de un modelo de aprendizaje y, por tanto, requerirán la parametrización de estos. Los parámetros e hiperparámetros del modelo dependerán de la aproximación y modelo que elijamos.
</p>
</center>

---
<center >
<p style="width:80%">
    <b>Epsilon</b>: Parámetro de exploración para los algoritmos basados en funciones valor (por ejemplo, los deep Q-learning). Puede tomar valores entre [0,1].
</p> 
</center>
<center>
<p style="width:80%">
    <b>Min Epsilon</b>: Valor mínimo del epsilon por si se quiere dejar aleatoriedad durante procesos de entrenamiento muy largos en los que el <b>epsilon</b> decaería hasta cero.
</p> 
</center>
<center>
<p style="width:80%">
    <b>Epsilon_decay</b>: Valor numérico mediante el cual se reduce el valor de epsilon cada determinado número de episodios.
</p> 
</center>
<center>
<p style="width:80%">
    <b>Gamma</b>: Término de descuento para el cálculo de los retornos acumulados con descuento. Dado que nuestro MDP se define con tareas episódicas gamma podrá tomar valores en el intervalo [0,1].
</p> 
</center>
<center>
<p style="width:80%">
    <b>Alpha</b>: Step-size, término indicando el tamaño de las actualizaciones durante el aprendizaje. Dado que nuestro problema es no estacionario (el sistema evoluciona en el tiempo) el alpha será un valor estático o adaptativo pero nunca un valor que decaiga a cero con el tiempo.
</p> 
</center>

<center>
<p style="width:80%">
    <b>Buffer Size</b>: Tamaño de la memoria encargada de obtener un conjunto de datos que aproximadamente sea independiente e idénticamente distribuido (i.i.d).
</p> 
</center>

---
<center>
<p style="width:80%">
<b>*Nota*</b>: Los parámetros que definen la arquitectura de la red dependen de lo flexible y compleja que se construyan en los agentes. Los agentes ofrecidos para el taller mantienen la filosofía del mismo y se mantendrán sencillos.
</p> 
</center>
<center>
<p style="width:80%">
    <b>Batch size</b>: Tamaño de los paquetes de datos que se envían al modelo durante su aprendizaje.
</p> 
</center>

<center>
<p style="width:80%">
    <b>Hidden_neurons</b>: Número de neuronas en las capas ocultas de los modelos.
</p> 
</center>

---
<center>
<p style="width:80%">
    <b>Nepisodes</b>: Número de sesiones de trading (episodios) que se van a realizar para el entrenamiento del modelo.
</p> 
</center>
<center>
<p style="width:80%">
    <b>N_log</b>: Valor numérico indicando cada cuántos episodios se realiza un log de la situación actual del aprendizaje.
</p> 
</center>
<center>
<p style="width:80%">
    <b>Learn_after</b>: Valor numérico indicando cada cuántos steps se realiza un aprendizaje por parte de las redes.
</p> 
</center>
<center>
<p style="width:80%">
    <b>Update_target_after</b>: Valor numérico indicando cada cuántos steps se copian los pesos de la red primary a la red target.
</p> 
</center>

In [ ]:
"""
    Agent Params
"""
# TODO: Selecciona un epsilon inicial para el entrenamiento
epsilon = "-----"
# TODO: Selecciona un min_epsilon para el entrenamiento
min_epsilon = "-----"
# TODO: Selecciona un gamma para el aprendizaje
gamma = "-----"
# TODO: Selecciona un alpha para el aprendizaje
alpha = "-----"
# TODO: Selecciona un buffer_size para el aprendizaje
buffer_size = "-----"
# TODO: Selecciona un batch_size para el aprendizaje
batch_size = "-----"
# TODO: Selecciona el número de neuronas para el modelo
hidden_neurons = "-----"

"""
    Training Params
"""
# TODO: Selecciona el número de episodios
nepisodes = "-----"
n_log = 25
#TODO: Determina el epsilon_decay para el proceso de entrenamiento
epsilon_decay = "-----"
learn_after = batch_size
# Cada cuántos pasos se copian los pesos del primary al target
update_target_after = 10 * learn_after

env = BestExecutionEnv(data, EnvironmentConfig(look_back=60), VwapRewardStrategy())

<center><h3>Inicialización del agente</h3></center>

In [ ]:
agent = "-----"

<center><h3>Recolección de los Episodios para el Buffer</h3></center>
<center>
<p style="width:80%">
Antes de empezar el entrenamiento, para que el modelo tenga experiencias de las que aprender y poder tomar acciones greedy con cierto criterio ha de tener muestras para entrenar. Para ello vamos a llenar el buffer con experiencias siguiendo una política estocástica con el objetivo de explorar el entorno inicialmente.
</p>

In [ ]:
# En este punto eps es 1 -> actuando random
s = env.reset()
for exps in range(buffer_size):  
    a = agent.act(s)
    s1, r, done, _ = env.step(a)
    agent.experience(s, a, r, s1, done)
    s = s1
    if not exps % 10000:
        print(f'buffer exps: {exps}')
    if done:
        s = env.reset()

#### Train Algo

In [ ]:
agent.set_trainable(True)
learn_counter = 0
history_steps = []
history_rewards = []
history_disc_rewards = []
history_losses = []
for episode in range(nepisodes):
    s = env.reset()
    step = 0
    cum_reward = 0
    dis_cum_reward = 0
    episode_losses = []
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        agent.experience(s, a, r, s1, done)
        learn_counter += 1
        cum_reward += r
        dis_cum_reward += agent.gamma ** step * r
        s = s1
        step += 1
        if not learn_counter % learn_after:
            mse = agent.learn()
        if not learn_counter % update_target_after:
            agent.update_target()
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            history_rewards.append(cum_reward)
            history_disc_rewards.append(dis_cum_reward)
            history_losses.append(mse)
            history_steps.append(step)
            if not episode % n_log:
                mse = agent.learn()
                print(
                    f'Episode: {episode}, '
                    f'steps: {np.round(np.mean(history_steps[-n_log:]), 2)}, '
                    f'rew: {np.round(np.mean(history_rewards[-n_log:]), 2)}, '
                    f'mse: {np.round(mse)}, '
                    f'eps: {np.round(agent.epsilon, 2)}'
                )
            break

#### Plot Results

In [ ]:
agent.set_trainable(False)
cum_reward = 0
step = 0
env = BestExecutionEnv(
    dict_["test"], EnvironmentConfig(look_back=60), VwapRewardStrategy()
)
s = env.reset()
while True:
    a = agent.act(s)
    s, r, done, _ = env.step(a)
    step += 1
    cum_reward += agent.gamma ** step * r
    if done:
        break
plot_results(env)

In [ ]:
pd.DataFrame(history_rewards).rolling(20).mean().plot()